In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Loading Data

In [ ]:
df = pd.read_csv("/kaggle/input/iris-dataset/iris.csv")

# Data Description

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df["species"].value_counts()

In [ ]:
df.describe()

# Data Visualization

In [ ]:
sns.pairplot(df)
plt.show()

In [ ]:
# --- Count Plot ---
plt.figure(figsize=(8, 5))
sns.countplot(x='species', data=df, palette='viridis')

# İngilizce başlık ve eksen isimleri
plt.title('Species Counts in the Dataset')
plt.xlabel('Species Name')
plt.ylabel('Count')

plt.show()

In [ ]:
# --- Correlation Heatmap ---
plt.figure(figsize=(10, 8))
correlation_matrix = df.select_dtypes(include=['float64', 'int64']).corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap Between Features')
plt.show()

In [ ]:

features = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']

# 2. Create a figure with 2 rows and 2 columns
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Flatten the axes array for easy iteration in a loop
axes = axes.flatten()

# 3. Create KDE plots for each feature using a loop
for i, feature in enumerate(features):
    sns.kdeplot(
        data=df, 
        x=feature, 
        hue="species", 
        fill=True, 
        palette="husl", 
        ax=axes[i] # Assigns the plot to the specific subplot box
    )
    axes[i].set_title(f'Distribution of {feature}', fontsize=14, fontweight='bold')
    axes[i].set_xlabel('Value (cm)')
    axes[i].set_ylabel('Density')

# Adjust layout to prevent overlapping
plt.tight_layout()
plt.suptitle('Iris Dataset: Distribution of All Features by Species', fontsize=18, y=1.02)
plt.show()

In [ ]:
# --- scatter plot ---
sns.scatterplot(x=df["sepal_length"], y=df["sepal_width"], hue=df["species"])
plt.show()

In [ ]:
sns.scatterplot(x=df["petal_length"], y=df["petal_width"], hue=df["species"])
plt.show()

# Encoding 

In [ ]:
# --- Label encoding ---
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
df["species"] = label_encoder.fit_transform(df["species"])

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df["species"].value_counts()

In [ ]:
X = df.drop("species", axis=1)
y = df["species"]

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=15)

# Logistic Regression Classification

In [ ]:
# --- Logicstic Regression ---
from sklearn.linear_model import LogisticRegression
model=LogisticRegression()
#hyperparameter tuning with class weights to handle imbalance
penalty=['l1', 'l2', 'elasticnet']
c_values=[100,10,1.0,0.1,0.01]
solver=['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga']
class_weight=[{0:w,1:y} for w in [1,10,50,100] for y in [1,10,50,100]]

params=dict(penalty=penalty,C=c_values,solver=solver,class_weight=class_weight)

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold
cv=StratifiedKFold()
grid=GridSearchCV(estimator=model,param_grid=params,scoring='accuracy',cv=cv)
grid.fit(X_train,y_train)

In [ ]:
grid.best_params_

In [ ]:
y_pred=grid.predict(X_test)
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
score=accuracy_score(y_pred,y_test)
print("score: ", score)
print(classification_report(y_pred,y_test))
print("confusion matrix: \n " , confusion_matrix(y_pred,y_test))

# Support Vector Classification (SVC)

In [ ]:
# --- SVC ---
from sklearn.svm import SVC
svc=SVC(kernel='linear')
svc.fit(X_train,y_train)

In [ ]:
y_pred2=svc.predict(X_test)

from sklearn.metrics import classification_report,confusion_matrix

score=accuracy_score(y_pred2,y_test)
print("score: ", score)
print(classification_report(y_test,y_pred2))
print(confusion_matrix(y_test,y_pred2))

In [ ]:
from sklearn.model_selection import GridSearchCV
 
# defining parameter range
param_grid = {'C': [0.1, 1, 10, 100, 1000],
              'gamma': [1, 0.1, 0.01, 0.001, 0.0001],
              'kernel': ['rbf']}

grid=GridSearchCV(SVC(),param_grid=param_grid,refit=True,cv=5)
grid.fit(X_train,y_train)

In [ ]:
grid.best_params_

In [ ]:
y_pred2=grid.predict(X_test)
score=accuracy_score(y_pred2,y_test)
print("score: ", score)
print(classification_report(y_test,y_pred2))
print(confusion_matrix(y_test,y_pred2))

# Naive Bayes Classification

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
gnb = GaussianNB()

gnb.fit(X_train, y_train)
y_pred3 = gnb.predict(X_test)

print("confusion matrix: \n", confusion_matrix(y_pred3, y_test))
print("accuracy score: ", accuracy_score(y_pred3, y_test))
print("classification report: ", classification_report(y_pred3, y_test))